# TinyLlama-1.1B SRD — `.axm` → GGUF Q4_K_M + MET Sidecar

Takes a TinyLlama-1.1B SRD `.axm` container and produces two deployment artefacts:

| Output | Size | Use |
|---|---|---|
| `tinyllama_1b_q4km.gguf` | ~636 MB | llama.cpp / PocketPal / llama-server |
| `tinyllama_1b_met_sidecar.json` | ~8 KB | MET hydration budget reference |

**Measured baselines (mobile — PocketPal, 6 CPU threads + GPU layers=99)**

| Metric | Value |
|---|---|
| GGUF Q4_K_M size | 636 MB |
| TG speed (mobile) | 12.03 t/s |
| PP speed (mobile) | 106.23 t/s |
| PPL Q4_K_M (wikitext-2) | 19.385 ±0.399 |

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

> **Patent pending — Orivael Inc.**  
> The SRD container format and signing protocol are the subject of pending patent claims.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Setup: clone axiom · install deps · persist AXIOM_MASTER_KEY
#           clone + cmake-build llama.cpp (GGML_CUDA=ON)  (~10–15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path("/content/axiom")
OUTPUT_DIR = Path("/content/axiom_output")
LLAMACPP   = Path("/content/llama.cpp")
BRANCH     = "claude/srd-prototype-benchmark-JRtv1"
REPO_URL   = "https://github.com/orivael-dev/axiom.git"
KEY_FILE   = OUTPUT_DIR / "axiom_master.key"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU info ──────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb   = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor   # 75=T4, 80=A100, 90=H100
print(f'GPU  : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
try:
    import psutil
    print(f'RAM  : {psutil.virtual_memory().total / 1024**3:.1f} GB system')
except ImportError:
    pass

# TinyLlama-1.1B FP16 is ~2.2 GB — fits on any Colab GPU
print('  ✓ TinyLlama-1.1B fits on any Colab GPU tier (FP16 ~2.2 GB)')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(AXIOM_DIR)], check=True)
    print(f'✓ axiom cloned  ({BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(AXIOM_DIR), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom updated  {r.stdout.strip()}')
sys.path.insert(0, str(AXIOM_DIR))

# ── pip install ───────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'psutil',
                'huggingface_hub', 'sentencepiece', 'safetensors', 'tqdm'],
               check=True)
print('✓ pip packages installed')

# ── Persist AXIOM_MASTER_KEY ──────────────────────────────────────────────────
# Save and restore across cell re-runs — regenerating would invalidate
# any .axm signatures produced in Cell 2/3.
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Clone llama.cpp ───────────────────────────────────────────────────────────
if not LLAMACPP.is_dir():
    print('\nCloning llama.cpp ...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp',
                    str(LLAMACPP)], check=True)
    print('✓ llama.cpp cloned')
else:
    print('✓ llama.cpp already present')

# ── cmake build with CUDA ─────────────────────────────────────────────────────
Q_BIN   = LLAMACPP / 'build/bin/llama-quantize'
CLI_BIN = LLAMACPP / 'build/bin/llama-cli'

if not Q_BIN.is_file():
    print(f'\nBuilding llama.cpp (CUDA arch={cuda_arch}) — ~10 min ...')
    t0 = time.time()
    subprocess.run(
        ['cmake', '-B', 'build',
         '-DCMAKE_BUILD_TYPE=Release',
         '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMACPP, check=True,
    )
    subprocess.run(
        ['cmake', '--build', 'build',
         '--target', 'llama-quantize', 'llama-cli',
         f'-j{os.cpu_count() or 4}'],
        cwd=LLAMACPP, check=True,
    )
    print(f'✓ llama.cpp built in {(time.time()-t0)/60:.1f} min')
else:
    print('✓ llama-quantize already built')

assert Q_BIN.is_file(),   f'llama-quantize not found at {Q_BIN}'
assert CLI_BIN.is_file(), f'llama-cli not found at {CLI_BIN}'
print(f'  llama-quantize : {Q_BIN}')
print(f'  llama-cli      : {CLI_BIN}')
print('\n✓ Cell 1 complete — proceed to Cell 2')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Source the .axm
#
# Set AXM_PATH to your existing .axm file path, or leave empty to pack
# TinyLlama-1.1B from scratch via the HuggingFace download + SRD-4 pack
# pipeline (~5 min on T4).  The .axm will be written to OUTPUT_DIR.
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

try:
    AXIOM_DIR
except NameError:
    AXIOM_DIR  = Path('/content/axiom')
    OUTPUT_DIR = Path('/content/axiom_output')
    sys.path.insert(0, str(AXIOM_DIR))

# ── ▼▼▼ SET YOUR PATH HERE, or leave empty to pack from scratch ▼▼▼ ──────────
AXM_PATH = ""   # ← set to your .axm path, or leave empty to pack from scratch
# ── ▲▲▲ ──────────────────────────────────────────────────────────────────────

PACK_SCRIPT = AXIOM_DIR / 'research' / 'quant' / 'pack_to_axm.py'
MODEL_ID    = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
MODEL_DIR   = OUTPUT_DIR / 'tinyllama_1b'
DEFAULT_AXM = OUTPUT_DIR / 'tinyllama_1b_srd4.axm'
STATS_JSON  = OUTPUT_DIR / 'tinyllama_1b_pack_stats.json'

if not AXM_PATH:
    print('AXM_PATH not set — packing TinyLlama-1.1B from scratch (~5 min on T4)')
    print()

    # Step A: download TinyLlama-1.1B-Chat-v1.0 from HuggingFace (no gating)
    if not MODEL_DIR.is_dir():
        print(f'Downloading {MODEL_ID} (~2.2 GB) ...')
        from huggingface_hub import snapshot_download
        snapshot_download(
            MODEL_ID,
            local_dir=str(MODEL_DIR),
            ignore_patterns=['*.bin'],   # prefer safetensors
        )
        print(f'✓ Downloaded → {MODEL_DIR}')
    else:
        print(f'✓ Model already present: {MODEL_DIR}')

    # Step B: SRD-4 pack (W4, α=0, ~4.5 bpw)
    if DEFAULT_AXM.is_file():
        print(f'✓ AXM already exists: {DEFAULT_AXM}')
    else:
        print('\nPacking TinyLlama with SRD-4 (W4, α=0, ~4.5 bpw) ...')
        assert PACK_SCRIPT.is_file(), f'pack_to_axm.py not found at {PACK_SCRIPT} — re-run Cell 1'
        t0 = time.time()
        r = subprocess.run([
            sys.executable, str(PACK_SCRIPT),
            '--model',      str(MODEL_DIR),
            '--srd4',
            '--group-size', '64',
            '--output',     str(DEFAULT_AXM),
            '--stats-json', str(STATS_JSON),
        ], cwd=str(AXIOM_DIR))
        elapsed = time.time() - t0
        if r.returncode != 0:
            raise RuntimeError(f'SRD pack failed (rc={r.returncode})')
        size_mb = DEFAULT_AXM.stat().st_size / 1024**2
        print(f'✓ {DEFAULT_AXM.name}  ({size_mb:.0f} MB)  {elapsed/60:.1f} min')

    AXM_PATH = str(DEFAULT_AXM)

AXM_PATH = Path(AXM_PATH)
assert AXM_PATH.is_file(), f'.axm not found at {AXM_PATH}'
size_mb = AXM_PATH.stat().st_size / 1024**2
print(f'\n✓ .axm ready : {AXM_PATH}  ({size_mb:.0f} MB)')
print('\n✓ Cell 2 complete — proceed to Cell 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Verify .axm proof ledger, then extract → GGUF Q4_K_M
#
# 1. axm_cli.py verify  — checks every HMAC-SHA256 proof shard
# 2. axm_to_gguf.py     — reconstructs FP16 → convert_hf_to_gguf.py
#                          → model_f16.gguf → llama-quantize Q4_K_M
#
# Time: ~10–15 min on T4 (FP16 reconstruction + quantize step)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

try:
    AXIOM_DIR
except NameError:
    AXIOM_DIR  = Path('/content/axiom')
    OUTPUT_DIR = Path('/content/axiom_output')
    LLAMACPP   = Path('/content/llama.cpp')
    sys.path.insert(0, str(AXIOM_DIR))

try:
    _axm = AXM_PATH
except NameError:
    raise RuntimeError('AXM_PATH not set — run Cell 2 first')

GGUF_PATH    = OUTPUT_DIR / 'tinyllama_1b_q4km.gguf'
AXM_TO_GGUF  = AXIOM_DIR / 'research' / 'quant' / 'axm_to_gguf.py'
EXTRACT_STATS = OUTPUT_DIR / 'tinyllama_1b_extract_stats.json'

assert _axm.is_file(),   f'.axm not found at {_axm} — run Cell 2 first'
assert (LLAMACPP / 'build/bin/llama-quantize').is_file(), \
    'llama-quantize not found — re-run Cell 1'

# ── Step 1: Verify proof ledger ───────────────────────────────────────────────
print(f'Verifying {_axm.name} ...')
result = subprocess.run(
    [sys.executable, str(AXIOM_DIR / 'axm_cli.py'), 'verify', str(_axm)],
    cwd=str(AXIOM_DIR), capture_output=True, text=True,
)
try:
    verify_data = json.loads(result.stdout)
except json.JSONDecodeError:
    verify_data = {'verified': False,
                   'error': result.stdout[-400:] + result.stderr[-200:]}

ok     = verify_data.get('verified', False)
FINGERPRINT = verify_data.get('fingerprint', '')
proofs = verify_data.get('proofs_checked', '?')

print(f'  {"✓ VERIFIED" if ok else "✗ FAILED"}')
print(f'  fingerprint    : {FINGERPRINT}')
print(f'  proofs checked : {proofs}')

if not ok:
    print(f'  error: {verify_data.get("error")}')
    raise RuntimeError('Verification failed — .axm may have been tampered with; do not extract')

# ── Step 2: Extract .axm → GGUF Q4_K_M ───────────────────────────────────────
if GGUF_PATH.is_file():
    print(f'\n✓ GGUF already exists: {GGUF_PATH}')
else:
    print('\nExtracting .axm → GGUF Q4_K_M ...')
    print(f'  Input  : {_axm}')
    print(f'  Output : {GGUF_PATH}')
    print(f'  Quant  : Q4_K_M  (target ~636 MB)')
    print()
    t0 = time.time()
    r = subprocess.run([
        sys.executable, str(AXM_TO_GGUF),
        '--container', str(_axm),
        '--gguf-out',  str(GGUF_PATH),
        '--llamacpp',  str(LLAMACPP),
        '--quant',     'Q4_K_M',
        '--device',    'cuda',
        '--stats-json', str(EXTRACT_STATS),
    ], cwd=str(AXIOM_DIR))
    elapsed = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError('GGUF extraction failed')
    print(f'\n✓ Extraction complete in {elapsed/60:.1f} min')

size_mb = GGUF_PATH.stat().st_size / 1024**2
print(f'  GGUF : {GGUF_PATH.name}  {size_mb:.0f} MB')
print(f'  vs measured baseline: 636 MB  (delta {size_mb - 636:+.0f} MB)')
if FINGERPRINT:
    print(f'  fingerprint : {FINGERPRINT}')
print('\n✓ Cell 3 complete — proceed to Cell 4')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Generate MET RAM sidecar
#
# Calls met_ram_estimator.py CLI with the TinyLlama-1.1B architecture flags.
# Writes tinyllama_1b_met_sidecar.json to OUTPUT_DIR.
#
# TinyLlama-1.1B architecture:
#   vocab_size: 32000  |  hidden_size: 2048  |  num_layers: 22
#   num_heads: 32      |  num_kv_heads: 4    |  intermediate_size: 5632
#   mlp_style: swiglu  |  bpw: 4.85 (Q4_K_M)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

try:
    AXIOM_DIR
except NameError:
    AXIOM_DIR  = Path('/content/axiom')
    OUTPUT_DIR = Path('/content/axiom_output')
    sys.path.insert(0, str(AXIOM_DIR))

SIDECAR_PATH = OUTPUT_DIR / 'tinyllama_1b_met_sidecar.json'
MET_SCRIPT   = AXIOM_DIR / 'research' / 'quant' / 'met_ram_estimator.py'

assert MET_SCRIPT.is_file(), f'met_ram_estimator.py not found at {MET_SCRIPT}'

# TinyLlama-1.1B architecture constants
ARCH = {
    'vocab_size':        32000,
    'hidden_size':       2048,
    'num_layers':        22,
    'num_heads':         32,
    'num_kv_heads':      4,
    'intermediate_size': 5632,
    'mlp_style':         'swiglu',
    'bpw':               4.85,
}

print('Generating MET sidecar for TinyLlama-1.1B ...')
print(f'  Architecture : {ARCH["num_layers"]} layers · '
      f'hidden {ARCH["hidden_size"]} · vocab {ARCH["vocab_size"]:,}')
print(f'  Attention    : {ARCH["num_heads"]} heads · {ARCH["num_kv_heads"]} KV heads (GQA)')
print(f'  MLP style    : {ARCH["mlp_style"]}  intermediate {ARCH["intermediate_size"]}')
print(f'  BPW          : {ARCH["bpw"]}  (Q4_K_M)')
print()

r = subprocess.run([
    sys.executable, str(MET_SCRIPT),
    '--model-id',          'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    '--vocab-size',        str(ARCH['vocab_size']),
    '--hidden-size',       str(ARCH['hidden_size']),
    '--num-layers',        str(ARCH['num_layers']),
    '--num-heads',         str(ARCH['num_heads']),
    '--num-kv-heads',      str(ARCH['num_kv_heads']),
    '--intermediate-size', str(ARCH['intermediate_size']),
    '--mlp-style',         ARCH['mlp_style'],
    '--bpw',               str(ARCH['bpw']),
    '--output',            str(SIDECAR_PATH),
], cwd=str(AXIOM_DIR), capture_output=False)

if r.returncode != 0:
    raise RuntimeError('met_ram_estimator.py failed')

assert SIDECAR_PATH.is_file(), f'Sidecar not written to {SIDECAR_PATH}'
sidecar = json.loads(SIDECAR_PATH.read_text())

# ── Print key numbers table ───────────────────────────────────────────────────
emb_mb   = sidecar['embedding_slot']['mb']
ir       = sidecar['intent_ram_budget']
peak_mb  = sidecar['peak_harm_mb']
inf_mb   = ir['INFORM']['total_mb']
savings  = (peak_mb - inf_mb) / peak_mb * 100

print()
print('KEY NUMBERS  (TinyLlama-1.1B Q4_K_M)')
print(f'  {"Metric":<38}  {"Value":>12}')
print('  ' + '─' * 52)
print(f'  {"Embedding slot (F16, always pinned)":<38}  {emb_mb:>10.1f} MB')
print(f'  {"INFORM budget (early chunk only)":<38}  {inf_mb:>10.1f} MB')
print(f'  {"Peak HARM budget (all chunks)":<38}  {peak_mb:>10.1f} MB')
print(f'  {"Savings: INFORM vs peak":<38}  {savings:>10.1f} %')
print(f'  {"GGUF size est. (embed Q8_0 + xfmr)":<38}  {sidecar["gguf_mb_est"]:>10} MB')
print(f'  {"Measured GGUF (mobile baseline)":<38}  {636:>10} MB')
print()
print('TRANSFORMER CHUNKS')
print(f'  {"Slot":<14}  {"Layers":<10}  {"MB":>7}  {"Precision"}')
print('  ' + '─' * 44)
print(f'  {"embedding":<14}  {"—embed—":<10}  {emb_mb:>7.1f}  F16  (always pinned)')
for slot, info in sidecar['transformer_chunks'].items():
    print(f'  {slot:<14}  {info["layers"]:<10}  {info["mb"]:>7.1f}  Q4_K_M')
print()
print(f'  Sidecar written → {SIDECAR_PATH}')
print(f'  Size            : {SIDECAR_PATH.stat().st_size / 1024:.1f} KB')
print('\n✓ Cell 4 complete — proceed to Cell 5')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Summary: output files + llama.cpp run command
# ══════════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path('/content/axiom_output')

GGUF_PATH    = OUTPUT_DIR / 'tinyllama_1b_q4km.gguf'
SIDECAR_PATH = OUTPUT_DIR / 'tinyllama_1b_met_sidecar.json'

def _size(p):
    if not Path(p).is_file():
        return 'missing'
    s = Path(p).stat().st_size
    return f'{s/1024**2:.0f} MB' if s > 1024*1024 else f'{s/1024:.1f} KB'

print('═' * 68)
print('  TINYLLAMA-1.1B SRD → GGUF  —  OUTPUT SUMMARY')
print('═' * 68)
print()
print(f'  {"File":<44}  {"Size":>10}')
print(f'  {"─"*44}  {"─"*10}')
print(f'  {"tinyllama_1b_q4km.gguf":<44}  {_size(GGUF_PATH):>10}')
print(f'  {"tinyllama_1b_met_sidecar.json":<44}  {_size(SIDECAR_PATH):>10}')

# ── MET budget snapshot ───────────────────────────────────────────────────────
if SIDECAR_PATH.is_file():
    sc   = json.loads(SIDECAR_PATH.read_text())
    ir   = sc['intent_ram_budget']
    emb  = sc['embedding_slot']['mb']
    peak = sc['peak_harm_mb']
    print()
    print('  MET HYDRATION BUDGETS')
    print(f'  {"Intent":<10}  {"Loaded chunks":<30}  {"RAM MB":>8}  {"UFS ms":>8}')
    print('  ' + '─' * 62)
    for intent, entry in ir.items():
        chunks_str = '+'.join(entry['chunks'])
        print(f'  {intent:<10}  {chunks_str:<30}  '
              f'{entry["total_mb"]:>6.1f} MB  {entry["ufs_load_ms"]:>5.1f} ms')
    print()
    print(f'  Embedding floor (between METs) : {emb:.1f} MB  (F16, always pinned)')
    print(f'  Peak (HARM / DECEIVE)          : {peak:.1f} MB')

# ── Measured mobile baseline ──────────────────────────────────────────────────
print()
print('  MOBILE BASELINE  (PocketPal, PP=512, TG=128, r=3)')
print(f'  GGUF Q4_K_M size : 636 MB')
print(f'  TG speed         : 12.03 t/s')
print(f'  PP speed         : 106.23 t/s')
print(f'  PPL Q4_K_M       : 19.385 ±0.399  (wikitext-2, 100 chunks)')
print(f'  PPL F16          : 19.205 ±0.404  (reference — delta is statistical noise)')
print()
print('═' * 68)

# ── llama.cpp run command ─────────────────────────────────────────────────────
print()
print('Run inference (llama.cpp):')
print(f'  ./llama-cli -m tinyllama_1b_q4km.gguf -p "Once upon a time" -n 100')
print()
print('Or with GPU offload:')
print(f'  ./llama-cli -m tinyllama_1b_q4km.gguf --n-gpu-layers 99 \\')
print( '    -p "Once upon a time" -n 100')
print()

# ── Dashboard note ────────────────────────────────────────────────────────────
print('Dashboard:')
print('  The MET sidecar is the .axiom_meta.json-format budget file.')
print('  Commit it to results/ for the Axiom dashboard:')
print(f'    cp {SIDECAR_PATH} {Path("results/tinyllama_1b_met_sidecar.json")}')
print()

# ── Colab download helper ──────────────────────────────────────────────────────
try:
    from google.colab import files as _colab_files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print('Downloading output files ...')
    for path in [GGUF_PATH, SIDECAR_PATH]:
        if Path(path).is_file():
            print(f'  {Path(path).name} ...')
            _colab_files.download(str(path))
    print('✓ Downloads triggered')
else:
    print(f'Output directory: {OUTPUT_DIR}')

print()
print('✓ Pipeline complete.')
print(f'  GGUF    : {GGUF_PATH}')
print(f'  Sidecar : {SIDECAR_PATH}')